# Train Test Split: To avoid overfitting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from backtesting import Backtest, Strategy
from sklearn.model_selection import TimeSeriesSplit

In [2]:
# Load the dt
df = pd.read_csv('data/df_GOOG_alphabet.csv', index_col=0, parse_dates=['Date'])
df.tail()

,Close,High,Low,Open,Volume,change_tmr,change_tmr_direction
Date,,,,,,,
2025-02-10,188.199997,189.990005,187.610001,189.059998,16606000,-0.604046,DOWN
2025-02-11,187.070007,188.800003,186.080002,186.835007,13028100,-0.884439,DOWN
2025-02-12,185.429993,186.830002,183.630005,185.229996,17632300,1.304030,UP
2025-02-13,187.880005,187.990005,184.880005,185.929993,12729300,-0.540488,DOWN
2025-02-14,186.869995,188.149994,186.110001,186.830002,12673300,NaN,DOWN


In [3]:
# df.tail()
# Drop NaN
df.dropna(inplace=True)
df.isnull().sum()

Close                   0
High                    0
Low                     0
Open                    0
Volume                  0
change_tmr              0
change_tmr_direction    0
dtype: int64

In [4]:
target = df.change_tmr #numeric
explanatory = df[['Close', 'High', 'Low', 'Open', 'Volume']]

## Train Test Split

In [5]:
n_days = len(df.index); n_days

2395

In [6]:
n_days_split = int(n_days*0.7)
X_train, y_train = explanatory.iloc[:n_days_split], target.iloc[:n_days_split]
X_test, y_test = explanatory.iloc[n_days_split:], target.iloc[n_days_split:]

In [7]:
X_train

,Close,High,Low,Open,Volume
Date,,,,,
2015-08-07,31.651274,32.018952,31.372775,31.896893,28078000
2015-08-10,31.573057,32.056819,31.449451,31.859528,36184000
2015-08-11,32.920715,33.624185,32.596380,33.340206,100584000
2015-08-12,32.859936,33.130961,32.497735,33.035304,58734000
2015-08-13,32.704987,33.106046,32.466395,32.848073,36214000
...,...,...,...,...,...
2022-03-28,141.441788,141.468197,139.327543,140.180816,23774000
2022-03-29,142.737137,143.646376,141.973873,142.647964,28678000
2022-03-30,142.133789,142.966794,141.658994,142.358476,21046000


## Fit the model on train set

In [9]:
model_dt_split = DecisionTreeRegressor(max_depth=15, random_state=42)
model_dt_split.fit(X=X_train, y=y_train)

DecisionTreeRegressor(max_depth=15, random_state=42)

## Evaluate the model

In [11]:
y_pred = model_dt_split.predict(X_test)
mse = mean_squared_error(y_true=y_test, y_pred=y_pred);mse

5.844217848795477

In [12]:
y_pred_train = model_dt_split.predict(X_train)
mean_squared_error(y_true=y_train, y_pred=y_pred_train)

1.7516737476098327

X_train has much lower error than X_test

# Backtesting
## 1. Create the `Strategy`

In [14]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    
    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False
        self.model.fit(X=X_train, y=y_train)
        
    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass
        
        

## 2. Run the backtest on `test` data

In [15]:
bt = Backtest(X_test, Regression, cash=10000, commission=.002, exclusive_orders=True)

In [ ]:
results = bt.run(limit_buy=1, limit_sell=-5)

results.to_frame(name='Values').style

,Values
Start,2022-04-04 00:00:00
End,2025-02-13 00:00:00
Duration,1046 days 00:00:00
Exposure Time [%],23.643950
Equity Final [$],18834.097925
Equity Peak [$],20817.098108
Commissions [$],180.588364
Return [%],88.340979
Buy & Hold Return [%],31.266920
Return (Ann.) [%],24.843095


In [17]:
df_results_test = results.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values': 'Out of Sample (Test)'}, axis=1)
df_results_test

,Out of Sample (Test)
Start,2022-04-04 00:00:00
End,2025-02-13 00:00:00
Duration,1046 days 00:00:00
Exposure Time [%],23.64395
Equity Final [$],18834.097925
Equity Peak [$],20817.098108
Commissions [$],180.588364
Return [%],88.340979


## 3. Run the backtest on `train` data

In [18]:
bt = Backtest(X_train, Regression, cash=10000, commission=.002, exclusive_orders=True)

results = bt.run(limit_buy=1, limit_sell=-5)

df_results_train = results.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values': 'Out of Sample (Test)'}, axis=1)
df_results_train

,Out of Sample (Test)
Start,2015-08-07 00:00:00
End,2022-04-01 00:00:00
Duration,2429 days 00:00:00
Exposure Time [%],76.372315
Equity Final [$],87075.853583
Equity Peak [$],93269.185813
Commissions [$],2507.398707
Return [%],770.758536


We see that the return on train set is much higher than on test set.
## 4. Compare both backtests

In [19]:
df_results = pd.concat([df_results_test, df_results_train], axis=1)
df_results

,Out of Sample (Test),Out of Sample (Test)
Start,2022-04-04 00:00:00,2015-08-07 00:00:00
End,2025-02-13 00:00:00,2022-04-01 00:00:00
Duration,1046 days 00:00:00,2429 days 00:00:00
Exposure Time [%],23.64395,76.372315
Equity Final [$],18834.097925,87075.853583
Equity Peak [$],20817.098108,93269.185813
Commissions [$],180.588364,2507.398707
Return [%],88.340979,770.758536


# Walk Forward (Anchored & Unanchored)
* Preserve the time order of the data 
* cf. normal train-test split which is more suitable for static datasets:
    * Works well for independen and identically distributed (IID) data
    * But breaks the time order, as future data can be used for training, and leads to data leakage. Therefore, not ideal for Time-Series
* It's to start with an initial training set, then slide forward in time, expanding or rolling the window, so that the model is train on the past, and test on a later time segment
* Unanchored: rolling window
    * pros: reduce memory usage
    * cons: forget old data
* Anchored: expanding window
    * pros: retain all past data, capturing long-term trend
    * cons: increasing data size and old data might be irrelevant

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2395 entries, 2015-08-07 to 2025-02-13
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Close                 2395 non-null   float64
 1   High                  2395 non-null   float64
 2   Low                   2395 non-null   float64
 3   Open                  2395 non-null   float64
 4   Volume                2395 non-null   int64  
 5   change_tmr            2395 non-null   float64
 6   change_tmr_direction  2395 non-null   object 
dtypes: float64(5), int64(1), object(1)
memory usage: 149.7+ KB


In [11]:
df1 = df.drop(columns='change_tmr_direction'); df1.head()

,Close,High,Low,Open,Volume,change_tmr
Date,,,,,,
2015-08-07,31.651274,32.018952,31.372775,31.896893,28078000,-0.247732
2015-08-10,31.573057,32.056819,31.449451,31.859528,36184000,4.093648
2015-08-11,32.920715,33.624185,32.596380,33.340206,100584000,-0.184966
2015-08-12,32.859936,33.130961,32.497735,33.035304,58734000,-0.473778
2015-08-13,32.704987,33.106046,32.466395,32.848073,36214000,0.101967


## Anchored: expanding window
### Proof of concept

In [7]:
ts = TimeSeriesSplit(test_size=200) #n_split=5 by default, test size as 200 days

Conduct 5 times of split between 2015-2025.

In [12]:
splits = ts.split(X=df1) #save generator into variable

In [13]:
split1 = next(splits); split1

(array([   0,    1,    2, ..., 1392, 1393, 1394], shape=(1395,)),
 array([1395, 1396, 1397, 1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405,
        1406, 1407, 1408, 1409, 1410, 1411, 1412, 1413, 1414, 1415, 1416,
        1417, 1418, 1419, 1420, 1421, 1422, 1423, 1424, 1425, 1426, 1427,
        1428, 1429, 1430, 1431, 1432, 1433, 1434, 1435, 1436, 1437, 1438,
        1439, 1440, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449,
        1450, 1451, 1452, 1453, 1454, 1455, 1456, 1457, 1458, 1459, 1460,
        1461, 1462, 1463, 1464, 1465, 1466, 1467, 1468, 1469, 1470, 1471,
        1472, 1473, 1474, 1475, 1476, 1477, 1478, 1479, 1480, 1481, 1482,
        1483, 1484, 1485, 1486, 1487, 1488, 1489, 1490, 1491, 1492, 1493,
        1494, 1495, 1496, 1497, 1498, 1499, 1500, 1501, 1502, 1503, 1504,
        1505, 1506, 1507, 1508, 1509, 1510, 1511, 1512, 1513, 1514, 1515,
        1516, 1517, 1518, 1519, 1520, 1521, 1522, 1523, 1524, 1525, 1526,
        1527, 1528, 1529, 1530, 1531, 1532, 15

Data is split on the start of the data (about 2015-2019), and test set is 200 days afterwards.
* In the first split, the train set is taking about 86% of data, and test set about 14%.
* For the second split, the train set will expand another 200 days of data, including the old dataset.

In [17]:
list_of_train = []
list_of_test = []

for index_train, index_test in ts.split(df1):
    list_of_train.append(df1.iloc[index_train])
    list_of_test.append(df1.iloc[index_test])

In [18]:
list_of_train[0]

,Close,High,Low,Open,Volume,change_tmr
Date,,,,,,
2015-08-07,31.651274,32.018952,31.372775,31.896893,28078000,-0.247732
2015-08-10,31.573057,32.056819,31.449451,31.859528,36184000,4.093648
2015-08-11,32.920715,33.624185,32.596380,33.340206,100584000,-0.184966
2015-08-12,32.859936,33.130961,32.497735,33.035304,58734000,-0.473778
2015-08-13,32.704987,33.106046,32.466395,32.848073,36214000,0.101967
...,...,...,...,...,...,...
2021-02-16,105.715157,107.248648,104.841298,104.841298,22676000,0.301177
2021-02-17,106.034508,106.301052,104.570271,104.624079,21418000,-0.524751
2021-02-18,105.480995,106.254963,104.808911,105.141714,22432000,-0.764354


In [19]:
list_of_test[0]

,Close,High,Low,Open,Volume,change_tmr
Date,,,,,,
2021-02-23,103.172295,103.727801,99.742616,100.888006,33348000,1.160290
2021-02-24,104.383446,104.662944,101.541659,101.725994,24966000,-3.141253
2021-02-25,101.204361,104.368993,100.702662,103.002397,36568000,0.270025
2021-02-26,101.478378,103.179762,100.442101,102.158932,41670000,2.145073
2021-03-01,103.702881,103.952480,101.938720,102.457850,28090000,-0.273136
...,...,...,...,...,...,...
2021-11-30,141.941986,146.103532,141.557366,144.929512,41590000,-0.588905
2021-12-01,141.110977,145.974651,140.993403,143.696186,28476000,1.501288
2021-12-02,143.261749,144.157031,140.477244,141.316240,21250000,-0.881282


In [20]:
list_of_train[1]

,Close,High,Low,Open,Volume,change_tmr
Date,,,,,,
2015-08-07,31.651274,32.018952,31.372775,31.896893,28078000,-0.247732
2015-08-10,31.573057,32.056819,31.449451,31.859528,36184000,4.093648
2015-08-11,32.920715,33.624185,32.596380,33.340206,100584000,-0.184966
2015-08-12,32.859936,33.130961,32.497735,33.035304,58734000,-0.473778
2015-08-13,32.704987,33.106046,32.466395,32.848073,36214000,0.101967
...,...,...,...,...,...,...
2021-11-30,141.941986,146.103532,141.557366,144.929512,41590000,-0.588905
2021-12-01,141.110977,145.974651,140.993403,143.696186,28476000,1.501288
2021-12-02,143.261749,144.157031,140.477244,141.316240,21250000,-0.881282


In [21]:
list_of_test[1]

,Close,High,Low,Open,Volume,change_tmr
Date,,,,,,
2021-12-07,147.506485,147.769046,145.180842,145.427453,23258000,0.459932
2021-12-08,148.188049,148.622478,146.672985,147.800434,18964000,-0.414909
2021-12-09,147.575745,149.069371,147.000815,147.645486,18580000,0.382719
2021-12-10,148.142715,148.865111,146.829928,148.566197,21634000,-1.343183
2021-12-13,146.179260,148.030610,145.835996,147.912534,24104000,-1.196099
...,...,...,...,...,...,...
2022-09-16,103.258980,103.657549,101.490341,102.601347,64540100,0.211847
2022-09-19,103.478195,103.647585,102.003498,102.172888,19738600,-1.983702
2022-09-20,101.465424,102.800622,100.757966,102.511660,24001700,-1.819818


### Applied ML model

In [23]:
y = df1.change_tmr
X = df1.drop(columns='change_tmr')

In [ ]:
# list_of_train = []
# list_of_test = []

# for index_train, index_test in ts.split(df1):
#     X_train, y_train = X.iloc[index_train], y.iloc[index_train]
#     X_test, y_test = X.iloc[index_test], y.iloc[index_test]
    

In [ ]:
# model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)
# model_dt.fit(X_train, y_train)

# y_pred = model_dt.predict(X_test)
# mse = mean_squared_error(y_true=y_test, y_pred=y_pred); mse

10.279881369739929

But to get 5 different splits of the data, we need to apply to for loop:

In [26]:
model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)

mse_list = []

for index_train, index_test in ts.split(df1):
    X_train, y_train = X.iloc[index_train], y.iloc[index_train]
    X_test, y_test = X.iloc[index_test], y.iloc[index_test]
    
    model_dt.fit(X_train, y_train)

    y_pred = model_dt.predict(X_test)
    mse = mean_squared_error(y_true=y_test, y_pred=y_pred)
    mse_list.append(mse)
    
    
    

In [27]:
mse_list

[4.559796351225358,
 5.902505483511282,
 9.40283035708461,
 4.9416627596885645,
 10.279881369739929]

Calculate the average of the error:

In [28]:
np.mean(mse_list)

np.float64(7.017335264249948)

### Anchored WFV in Backtesting 

In [39]:
df1

,Close,High,Low,Open,Volume,change_tmr
Date,,,,,,
2015-08-07,31.651274,32.018952,31.372775,31.896893,28078000,-0.247732
2015-08-10,31.573057,32.056819,31.449451,31.859528,36184000,4.093648
2015-08-11,32.920715,33.624185,32.596380,33.340206,100584000,-0.184966
2015-08-12,32.859936,33.130961,32.497735,33.035304,58734000,-0.473778
2015-08-13,32.704987,33.106046,32.466395,32.848073,36214000,0.101967
...,...,...,...,...,...,...
2025-02-07,187.139999,193.014999,185.100006,192.740005,29565700,0.563229
2025-02-10,188.199997,189.990005,187.610001,189.059998,16606000,-0.604046
2025-02-11,187.070007,188.800003,186.080002,186.835007,13028100,-0.884439


In [46]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    n_train = 600
    coef_retrain = 200
    
    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False
        
        X_train = self.data.df.iloc[:self.n_train, :-1]#the first day till the last 600, and excpet the last column
        y_train = self.data.df.iloc[:self.n_train, -1]
        
        self.model.fit(X=X_train, y=y_train)
    
    # buy/sell action    
    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :-1] #excluding last column
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass
        
        

Create new class to include anchored walk forward procedure after the initial step of training with the first 600 days has started:

In [47]:
class WalkForwardAnchored(Regression):
    def next(self):
        # no action taken and move on to the following day
        if len(self.data) < self.n_train:
            return
        
        # retrain the model each 200 days
        if len(self.data) % self.coef_retrain == 0:
            X_train = self.data.df.iloc[:, :-1]
            y_train = self.data.df.iloc[:, -1]
            
            self.model.fit(X_train, y_train)
            super().next() #call the super class (ie. Regression.next)
        
        else:
            super().next()

In [48]:
bt = Backtest(df1, WalkForwardAnchored, cash=10000, commission=.002, exclusive_orders=True)

* `Backtest` auto-assigns `df1` to `self.data`, and use `.df` to access `.iloc`

In [49]:
stats_skopt, heatmap, optimize_result = bt.optimize(
    limit_buy = [0, 6],
    limit_sell = [-6, 0],
    maximize= 'Return [%]',
    method='skopt',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True
)

dff_anchored = heatmap.reset_index()
dff_anchored = dff_anchored.sort_values('Return [%]', ascending=False)

/var/folders/c7/zvfr__1n2v34xs_blw9dnt8h0000gn/T/ipykernel_18961/3012203600.py:1: DeprecationWarning: `Backtest.optimize(method="skopt")` is deprecated. Use `method="sambo"`.
  stats_skopt, heatmap, optimize_result = bt.optimize(


In [ ]:
# dff_anchored

In [ ]:
dff_anchored_pivot = dff_anchored.pivot(index='limit_buy', columns='limit_sell', values='Return [%]')
dff_anchored_pivot.sort_index(axis=1, ascending=False)

limit_sell,0,-1,-2,-3,-4,-5,-6
limit_buy,,,,,,,
0,NaN,-62.403106,5.194034,NaN,198.593171,NaN,260.264042
1,-93.495078,-57.322386,-46.103258,50.854540,52.367940,256.458117,236.392064
2,-89.479713,-69.198445,-64.158472,-18.349776,-29.863059,73.015324,46.522835
3,-88.190255,-82.219824,-82.309983,-38.658912,-47.032196,155.475326,NaN
4,NaN,-74.058478,-74.438744,16.984719,0.837589,174.768563,45.507548
5,NaN,-79.342112,-79.388587,-24.514514,NaN,111.913485,NaN
6,NaN,NaN,-100.000000,NaN,-100.000000,NaN,NaN


In [53]:
dff_anchored_pivot.sort_index(axis=1, ascending=False)\
    .style.format(precision=0)\
        .background_gradient(vmin=np.nanmin(dff_anchored_pivot), vmax=np.nanmax(dff_anchored_pivot))\
            .highlight_null(props='background-color: transparent; color: transparent')

limit_sell,0,-1,-2,-3,-4,-5,-6
limit_buy,,,,,,,
0,nan,-62,5,nan,199,nan,260
1,-93,-57,-46,51,52,256,236
2,-89,-69,-64,-18,-30,73,47
3,-88,-82,-82,-39,-47,155,nan
4,nan,-74,-74,17,1,175,46
5,nan,-79,-79,-25,nan,112,nan
6,nan,nan,-100,nan,-100,nan,nan


* Not too stricit limit on buy, but strick limit on sell.

## Unanchored: rolling window

* Save strategies class to a `py` file for easy access

In [54]:
import strategies

strategies.WalkForwardUnanchored

strategies.WalkForwardUnanchored

In [55]:
bt_unanchored = Backtest(df1, strategies.WalkForwardUnanchored, cash=10000, commission=.002, exclusive_orders=True)

In [56]:
stats_skopt, heatmap, optimize_result = bt_unanchored.optimize(
    limit_buy = [0, 6],
    limit_sell = [-6, 0],
    maximize= 'Return [%]',
    method='skopt',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True
)

dff_unanchored = heatmap.reset_index()
dff_unanchored = dff_unanchored.sort_values('Return [%]', ascending=False)

/var/folders/c7/zvfr__1n2v34xs_blw9dnt8h0000gn/T/ipykernel_18961/2304570296.py:1: DeprecationWarning: `Backtest.optimize(method="skopt")` is deprecated. Use `method="sambo"`.
  stats_skopt, heatmap, optimize_result = bt_unanchored.optimize(


In [ ]:
# dff_unanchored

In [58]:
dff_unanchored_pivot = dff_unanchored.pivot(index='limit_buy', columns='limit_sell', values='Return [%]')
dff_unanchored_pivot.sort_index(axis=1, ascending=False)

limit_sell,0,-1,-2,-3,-4,-5,-6
limit_buy,,,,,,,
0,NaN,-25.533441,74.431244,NaN,97.256537,NaN,212.998658
1,-91.235049,-50.095452,-48.210604,-43.209648,-34.886666,51.794883,102.250690
2,-89.835812,-61.810087,-55.211487,-52.636045,-18.098807,26.422412,113.939338
3,-84.187918,-57.943185,-65.015651,-63.048777,-8.241624,33.589023,113.939338
4,NaN,-72.550914,-78.211262,-80.406108,-19.564513,-25.904083,26.230390
5,NaN,-68.002259,-80.799781,-81.309283,NaN,26.397733,NaN
6,NaN,NaN,-74.563490,NaN,35.241275,NaN,NaN


In [59]:
dff_unanchored_pivot.sort_index(axis=1, ascending=False)\
    .style.format(precision=0)\
        .background_gradient(vmin=np.nanmin(dff_unanchored_pivot), vmax=np.nanmax(dff_unanchored_pivot))\
            .highlight_null(props='background-color: transparent; color: transparent')

limit_sell,0,-1,-2,-3,-4,-5,-6
limit_buy,,,,,,,
0,nan,-26,74,nan,97,nan,213
1,-91,-50,-48,-43,-35,52,102
2,-90,-62,-55,-53,-18,26,114
3,-84,-58,-65,-63,-8,34,114
4,nan,-73,-78,-80,-20,-26,26
5,nan,-68,-81,-81,nan,26,nan
6,nan,nan,-75,nan,35,nan,nan


* Unanchored doesn't necessarily perform much better, but the training time is much faster.

## Interpret the strategies' performance

In [60]:
bt.plot(filename='reports_backtesting/walk_forward_anchored.html')

GridPlot(id='p1318', ...)

In [61]:
bt_unanchored.plot(filename='reports_backtesting/walk_forward_unanchored.html')

GridPlot(id='p1665', ...)

# Conclusion:
Walk-forward-anchor:
* Final at 360%
* Peak at 398%

Walk-forward-unanchor:
* Final at 313%
* Peak at 346%

Both appraoches indicated start of investing around 2018, and the return results are very simliar.
